In [1]:
import pandas as pd
import re
import pysam
from collections import defaultdict

In [2]:
df = pd.read_csv('coverage_1mb.tsv', sep='\t', header=None, 
                 names=['sample', 'chr', 'start', 'end', 'reads'])

def extract_id(sample):
    match = re.search(r'_(\d+sib)_', sample)
    if match:
        return match.group(1)
    return sample.split('_')[-1]

df['short_name'] = df['sample'].apply(extract_id)

df.head()

,sample,chr,start,end,reads,short_name
0,sample,chr,start,end,reads,sample
1,202403171842_221101001_2P231208056US2S2716BX_A...,chr1,0,1000000,989,39sib
2,202403171842_221101001_2P231208056US2S2716BX_A...,chr1,1000000,2000000,2,39sib
3,202403171842_221101001_2P231208056US2S2716BX_A...,chr1,4000000,5000000,1749,39sib
4,202403171842_221101001_2P231208056US2S2716BX_A...,chr1,5000000,6000000,590,39sib


In [3]:
df['reads'] = pd.to_numeric(df['reads'], errors='coerce')
stats = df.groupby('short_name').agg(
    total_bins=('reads', 'count'),
    bins_ge_20x=('reads', lambda x: (x >= 20).sum()),
    fraction_ge_20x=('reads', lambda x: f"{(x >= 20).mean() * 100:.2f}%")
).reset_index()

In [4]:
stats.to_csv('sample_coverage_stats.csv', index=False)
print(stats)

   short_name  total_bins  bins_ge_20x fraction_ge_20x
0       10sib        2802         2423          86.47%
1       11sib        2695          706          26.20%
2       12sib           8            0           0.00%
3       13sib           7            0           0.00%
4       14sib          10            0           0.00%
5       15sib          36            4          11.11%
6       16sib           6            0           0.00%
7       17sib           8            0           0.00%
8       18sib           5            0           0.00%
9       19sib          11            0           0.00%
10      20sib           2            0           0.00%
11      22sib           1            0           0.00%
12      23sib           2            0           0.00%
13      24sib        2916         2723          93.38%
14      25sib        2571          775          30.14%
15      26sib        2917         2759          94.58%
16      27sib        2794         2379          85.15%
17      28

подсчет количества SNP в каждом бине

In [5]:
def count_snps_in_bins(vcf_path, bin_size=1_000_000):
    """
    Подсчитывает количество SNP в бинах указанного размера.
    
    Args:
        vcf_path (str): Путь к VCF файлу (.vcf или .vcf.gz).
        bin_size (int): Размер бина в парах оснований.
    
    Returns:
        pd.DataFrame: Таблица с колонками chr, start, end, snp_count.
    """
    vcf = pysam.VariantFile(vcf_path)
    bin_counts = defaultdict(int)
    bin_boundaries = {}
    
    for record in vcf.fetch():
        # Проверяем, что запись - это SNP, а не индел или другой тип
        # Тип SNP часто хранится в информации, но можно использовать и другие критерии
        if record.alts is None or len(record.alts) == 0:
            continue
            
        # Простейший способ: считаем, что SNP это замена одного нуклеотида на другой
        is_snp = all(len(alt) == 1 and len(record.ref) == 1 for alt in record.alts)
        if not is_snp:
            continue
        
        chrom = record.chrom
        # Фильтруем только основные хромосомы
        if chrom not in [f'chr{i}' for i in list(range(1,23)) + ['X', 'Y']]:
            continue
            
        pos = record.pos
        bin_id = (pos - 1) // bin_size
        start = bin_id * bin_size + 1
        end = (bin_id + 1) * bin_size
        
        key = (chrom, bin_id)
        bin_counts[key] += 1
        
        if key not in bin_boundaries:
            bin_boundaries[key] = (start, end)
    
    # Формируем результат в виде DataFrame
    data = []
    for (chrom, bin_id), count in bin_counts.items():
        start, end = bin_boundaries[(chrom, bin_id)]
        data.append({'chr': chrom, 'start': start, 'end': end, 'snp_count': count})
    
    df_result = pd.DataFrame(data)
    # Сортируем по хромосоме и позиции для удобства
    # Сортировка по хромосоме: chr1, chr2, ..., chrX, chrY
    df_result['chr_num'] = df_result['chr'].str.replace('chr', '').map(
        lambda x: int(x) if x.isdigit() else (23 if x == 'X' else 24)
    )
    df_result = df_result.sort_values(['chr_num', 'start']).drop(columns=['chr_num'])
    
    return df_result

In [6]:
df_result = count_snps_in_bins("/storage/dasha/output/sib1-56/202403171842_221101001_2P231208056US2S2716BX_A_202411_4sib_L01/202403171842_221101001_2P231208056US2S2716BX_A_202411_4sib_L01.vcf.gz")

In [7]:
def calculate_snp_metrics(df):
    """Возвращает словарь с метриками"""
    metrics = {
        'total_snps': df['snp_count'].sum(),
        'bins_with_snps': (df['snp_count'] > 0).sum(),
        'fraction_bins_with_snps': (df['snp_count'] > 0).mean() * 100,
        'mean_snps_per_bin': df['snp_count'].mean(),
        'median_snps_per_bin': df['snp_count'].median(),
        'cv_snps_per_bin': df['snp_count'].std() / df['snp_count'].mean(),  # коэф. вариации
    }
    
    # По хромосомам
    chrom_stats = df.groupby('chr')['snp_count'].agg(['mean', 'std', 'count'])
    
    return metrics, chrom_stats

metrics, chrom_stats = calculate_snp_metrics(df_result)

print(f"Всего SNP: {metrics['total_snps']:,}")
print(f"Доля бинов с SNP: {metrics['fraction_bins_with_snps']:.1f}%")
print(f"Среднее SNP на бин: {metrics['mean_snps_per_bin']:.1f}")

Всего SNP: 43,872
Доля бинов с SNP: 100.0%
Среднее SNP на бин: 16.1
